# Team allocation

A team of four people, eight tasks. Each task looks reasonable on its own. Let's see what happens when we put them on one schedule.

First, open an in-memory workspace and create the project. It starts on Monday 2026-10-05.

In [ ]:
from datetime import date

import project_planner as pp

ws = pp.open_workspace(":memory:")
ws.new_project("Team allocation", start=date(2026, 10, 5))

The team, with an hourly rate for each person.

In [ ]:
rates = {"Ana": "90", "Ben": "80", "Chloe": "70", "Dev": "60"}
people = {name: ws.add_resource(name, hourly_rate=rate) for name, rate in rates.items()}

Eight tasks of one or two weeks each. Every task has a duration, one or more people with a percentage of their time, and sometimes a predecessor. Read down the list: nothing looks unusual.

In [ ]:
plan = [
    # name, duration, {person: percent}
    ("Requirements", "5d", {"Ana": 100}),
    ("Architecture", "5d", {"Ana": 50, "Ben": 50}),
    ("UX design", "5d", {"Ana": 50, "Chloe": 100}),
    ("API build", "5d", {"Ben": 100}),
    ("Frontend build", "5d", {"Chloe": 100, "Ana": 50}),
    ("Test plan", "5d", {"Ana": 50}),
    ("Data migration", "5d", {"Ana": 100}),
    ("Release notes", "2d", {"Dev": 25}),
]
tasks = []
for name, duration, who in plan:
    task = ws.add_task(name, duration=duration)
    for person, percent in who.items():
        ws.set_assignment(task, people[person], percent)
    tasks.append(task)

# finish-to-start dependencies: (predecessor, successor), 1-based task numbers
for pred, succ in [(2, 4), (3, 5), (1, 7), (4, 8)]:
    ws.add_dependency(tasks[pred - 1], tasks[succ - 1], "FS")

Now calculate the schedule. The dates come only from the dependencies and durations.

In [ ]:
result = ws.schedule()
print(f"Project finish: {result.project_finish:%Y-%m-%d %H:%M}")
result

The loading view shows how busy one person is over time. Ana's tasks all start in week 1, so she is asked to work at 250% of her time, then 150% in week 2.

In [ ]:
ws.loading(people["Ana"])

Dev, on the other hand, has a single quarter-time task of two days.

In [ ]:
ws.loading(people["Dev"])

For every person: peak load, assigned person-days, utilization (assigned days divided by the project's working span) and cost. These come from the public read views.

In [ ]:
from decimal import Decimal

span = result.working_span_days
summary = {}
for name, rid in people.items():
    peak = max(s.percent for s in ws.loading(rid).segments)
    mine = [a for a in result.assignments.values() if a.resource_id == rid]
    days = sum((a.assignment_days for a in mine), Decimal(0))
    cost = sum((a.cost for a in mine), Decimal(0))
    utilization = (days / span * 100).quantize(Decimal("0.1"))
    summary[name] = peak
    print(f"{name}: peak {peak.normalize():f}% | {days.normalize():f} person-days | "
          f"utilization {utilization}% | cost {cost:.2f}")

print(f"Total cost: {result.total_cost:.2f}")
print(f"Project finish: {result.project_finish:%Y-%m-%d %H:%M}")

The same peaks as text bars. Each block is 25% of a person's time, and the `|` marks 100%. Anything to the right of it is more than one person can do.

In [ ]:
for name, peak in summary.items():
    blocks = int(peak // 25)
    bar = ("█" * min(blocks, 4)).ljust(4) + "|" + "█" * max(blocks - 4, 0)
    print(f"{name:<6} {bar} {peak.normalize():f}%")

Optional: the same chart with matplotlib, if it is installed.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.barh(list(summary), [float(p) for p in summary.values()])
    ax.axvline(100, linestyle="--", color="red")
    ax.invert_yaxis()
    ax.set_xlabel("Peak load (%)")
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping the chart")

## Takeaway

The schedule assumes everyone can do everything at once. Ana is booked at 250% while Dev is at 25%. Two ways out: **level** the plan so Ana's tasks run one after another (notebook 03), or **reassign** some of her work to people with room, such as Dev.